# ML Pipeline Reference
## A step-by-step guide for each ML task

## Table of Contents
1. [General ML Pipeline](##1-general-ml-pipeline)
2. [Linear Regression](##2-linear-regression)
3. [Binary Classification (Logistic Regression)](##3-binary-classification-logistic-regression)
4. [Multiclass Classification](##4-multiclass-classification)
5. [K-Nearest Neighbors (KNN)](##5-k-nearest-neighbors-knn)
6. [Regularization (Ridge, Lasso, ElasticNet)](##6--regularization-ridge-lasso-elasticnet)
7. [KMeans Clustering](##7-kmeans-clustering)
8. [TF-IDF (Text Feature Extraction)](##8-tf-idf-text-feature-extraction)
9. [PyTorch Neural Network (Regression)](##9-pytorch-neural-network-regression)
9b. [PyTorch Neural Network (Classification)](###-pytorch-neural-network-classification)

## 1. General ML Pipeline

Every ML project follows these steps regardless of the task:

1. **Load the data** — get the dataset into a pandas DataFrame
2. **Explore the data** — understand the structure, spot missing values, check distributions
3. **Clean the data** — handle missing values, drop useless columns
4. **Prepare features** — encode categorical variables, select features, split into X and y
5. **Split the data** — separate into training and test sets (80/20)
6. **Scale the features** — standardize to mean 0, SD 1
7. **Train the model** — fit the model on training data
8. **Evaluate the model** — measure performance on test data
9. **Interpret results** — understand what the metrics mean and where the model fails

### Universal Imports - Always needed 

In [ ]:
# Universal imports — always needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Linear Regression


**When to use:**
When you need to predict a continuous numerical value (price, salary, temperature) 
and the relationship between features and target is approximately linear.

**When NOT to use:**
- Target is a category (use classification instead)
- Relationship between features and target is clearly non-linear

**Imports:**
````python
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
````

**Steps:**

1. Load and explore data
   - Check df.shape, df.head(), df.describe(), df.isnull().sum()

2. Clean data
   - Drop useless columns: df.drop('column', axis=1)
   - Fill missing values: df['col'].fillna(df['col'].median())
   - Drop rows with missing values: df.dropna(subset=['col'])

3. Prepare features
   - X = df.drop('target_column', axis=1)
   - y = df['target_column']

4. Split data
   - X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

5. Scale features
   - scaler = StandardScaler()
   - X_train_scaled = scaler.fit_transform(X_train)
   - X_test_scaled = scaler.transform(X_test)

6. Train model
   - model = LinearRegression()
   - model.fit(X_train_scaled, y_train)

7. Evaluate model
   - y_pred = model.predict(X_test_scaled)
   - mse = mean_squared_error(y_test, y_pred)
   - rmse = np.sqrt(mse)
   - r2 = r2_score(y_test, y_pred)

8. Interpret results
   - RMSE: average prediction error in the same unit as the target. Lower is better.
   - R²: how much of the variation in the target the model explains. 
         Closer to 1 is better. Below 0.5 is weak.
   - Coefficients: model.coef_ — one per feature, shows direction and strength 
                   of each feature's influence (only comparable after scaling)
   - Intercept: model.intercept_ — baseline prediction when all features are 0
````

**Common issues and considerations:**

- **Multicollinearity:** if two features are strongly correlated their 
  coefficients become unreliable. Check with a correlation heatmap before 
  training. If found, remove one of the correlated features or apply 
  regularization (Ridge or Lasso regression).

- **Outliers:** Linear regression is sensitive to outliers because MSE 
  penalizes large errors heavily. Check with box plots before training. 
  If found, either remove them or consider a more robust model.

- **Feature scaling is required** before comparing coefficients. 
  Without scaling, coefficients reflect the scale of the feature, 
  not its actual importance.

- **Always fit scaler on training data only.** Fitting on test data 
  leaks information and makes evaluation unreliable.

- **Assumptions to check:**
  - Linearity: if relationship is non-linear, consider polynomial features 
    or a different model
  - No multicollinearity: if features are correlated, remove one or apply 
    regularization
  - Homoscedasticity: if errors grow with value size, consider log-transforming 
    the target variable
  - Independence: if data points are time-dependent, consider time series models

- **Convergence warning:** if sklearn raises a ConvergenceWarning increase 
  max_iter: LinearRegression(max_iter=1000). This means the model ran out 
  of iterations before finding the optimal parameters.

## 3. Binary Classification (Logistic Regression)

**When to use:**
When you need to predict one of two possible categories (yes/no, spam/not spam, 
survived/not survived) and the relationship between features and target is 
approximately linear.

**When NOT to use:**
- Target has more than two categories (use Multiclass Classification instead)
- Decision boundary is highly non-linear (use more complex models like SVM or 
  Random Forest)

**Imports:**
```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, roc_auc_score
```

**Steps:**

1. Load and explore data
   - Check df.shape, df.head(), df.describe(), df.isnull().sum()
   - Check class balance: df['target'].value_counts()
     If heavily imbalanced, accuracy alone will be misleading

2. Clean data
   - Drop useless columns: df.drop('column', axis=1)
   - Fill missing values: df['col'].fillna(df['col'].median())
   - Drop rows with missing values: df.dropna(subset=['col'])

3. Prepare features
   - Encode categorical features: pd.get_dummies(X, columns=['col'], drop_first=True)
   - X = df[['feature1', 'feature2', ...]]
   - y = df['target_column']

4. Split data
   - X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

5. Scale features
   - scaler = StandardScaler()
   - X_train_scaled = scaler.fit_transform(X_train)
   - X_test_scaled = scaler.transform(X_test)

6. Train model
   - model = LogisticRegression(random_state=42, max_iter=1000)
   - model.fit(X_train_scaled, y_train)

7. Evaluate model
   - y_pred = model.predict(X_test_scaled)
   - print(classification_report(y_test, y_pred))
   - cm = confusion_matrix(y_test, y_pred)
   - ConfusionMatrixDisplay(cm).plot()
   
   For ROC curve:
   - y_prob = model.predict_proba(X_test_scaled)[:, 1]
   - fpr, tpr, thresholds = roc_curve(y_test, y_prob)
   - auc = roc_auc_score(y_test, y_prob)
   - plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
   - plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')

8. Interpret results
   - Precision: of all predicted positives, how many were actually positive?
   - Recall: of all actual positives, how many did the model catch?
   - F1: harmonic mean of precision and recall. Best single metric when 
     classes are imbalanced.
   - AUC: overall model quality across all thresholds. Above 0.8 is good.
   - Confusion matrix: rows = actual, columns = predicted. 
     Diagonal = correct predictions. Off-diagonal = errors.

**Threshold adjustment:**
   - Default threshold is 0.5
   - Lower threshold → higher recall, lower precision (catches more positives,
     more false alarms)
   - Higher threshold → higher precision, lower recall (fewer false alarms, 
     misses more positives)
   - Use ROC curve to find the best operating point for your specific problem

- **Class imbalance:** if one class dominates, the model will be biased toward 
  it. Check with df['target'].value_counts(). Solutions: adjust class_weight 
  parameter: LogisticRegression(class_weight='balanced'), or adjust threshold.

- **Categorical features:** must be encoded before training. Use pd.get_dummies() 
  with drop_first=True to avoid the dummy variable trap.

- **Convergence warning:** increase max_iter if raised. Default is 100, 
  try 1000 for larger datasets.

- **Boolean features:** True/False values from get_dummies() are handled 
  automatically by sklearn but can be converted explicitly with .astype(int) 
  for clarity.

- **Always check both precision and recall,** not just accuracy. High accuracy 
  with imbalanced classes can be completely misleading.

## 4. Multiclass Classification


**When to use:**
When you need to predict one of three or more possible categories 
(digit 0-9, animal species, news article topic).

**When NOT to use:**
- Target is continuous (use regression instead)
- Target has only two categories (use binary classification instead)

**Imports:**
```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
```

**Steps:**

1. Load and explore data
   - Check df.shape, df.head(), df.describe(), df.isnull().sum()
   - Check class distribution: df['target'].value_counts()
   - For image data: visualize samples with plt.imshow()

2. Clean data
   - Same as binary classification
   - For image data: no cleaning usually needed, pixels are already numerical

3. Prepare features
   - X = data matrix (one row per sample, one column per feature)
   - y = target labels
   - For image data: pixels are already features, no encoding needed

4. Split data
   - Standard: train_test_split(X, y, test_size=0.2, random_state=42)
   - For benchmark datasets with conventional splits:
     X_train, X_test = X[:n], X[n:]
     y_train, y_test = y[:n], y[n:]

5. Scale features
   - Same as binary classification
   - Note: pixels with zero variance (always black) will have std=0 after 
     scaling. This is expected and harmless for border pixels in image data.

6. Train model
   - model = LogisticRegression(random_state=42, max_iter=1000)
   - model.fit(X_train_scaled, y_train)
   - Sklearn automatically uses OvR strategy for logistic regression

7. Evaluate model
   - y_pred = model.predict(X_test_scaled)
   - print(classification_report(y_test, y_pred))
   - cm = confusion_matrix(y_test, y_pred)
   - ConfusionMatrixDisplay(cm).plot()
   - plt.show()

8. Interpret results
   - Classification report shows precision, recall and F1 per class
   - Macro average: simple average across all classes, treats all equally
   - Weighted average: average weighted by number of samples per class,
     better when classes are imbalanced
   - Confusion matrix: NxN grid. Look for off-diagonal clusters to find 
     which classes are being confused with each other.

**OvR vs OvO strategies:**
   - OvR (One vs Rest): one classifier per class, picks highest confidence.
     Default for Logistic Regression.
   - OvO (One vs One): one classifier per pair of classes, picks by majority 
     vote. Better for algorithms that scale poorly with data size (e.g. SVM).
   - Sklearn chooses automatically — you rarely need to specify manually.

- **Training time:** multiclass problems with many features take significantly 
  longer to train. If you get a ConvergenceWarning increase max_iter.

- **Class imbalance:** same as binary — check value_counts() and use 
  class_weight='balanced' if needed.

- **Confused classes:** use the confusion matrix to identify which classes 
  the model struggles with. Similar looking classes (e.g. 5 and 3 in MNIST) 
  will have higher error rates. Solutions: more training data for those 
  classes, or a more powerful model.

- **Feature variance:** for high dimensional data like images, many features 
  may have zero variance and carry no information. Consider removing them 
  with variance thresholding before training.

- **ROC curve:** not directly applicable to multiclass. Use classification 
  report and confusion matrix instead.

## 5. K-Nearest Neighbors (KNN)

**When to use:**
- Small to medium sized datasets
- When you need a simple, interpretable baseline model
- When the decision boundary is non-linear
- Works for both classification and regression

**When NOT to use:**
- Large datasets — prediction is slow because it searches the entire 
  training set every time
- High dimensional data — distance measures become unreliable with 
  many features (curse of dimensionality)
- When features are on very different scales — always scale first

**Imports:**
```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor  # for regression
```

**Key concepts:**
- No training phase — memorizes the entire training set
- At prediction time: finds K closest points, majority vote (classification) 
  or average (regression)
- Distance measured with Euclidean distance by default
- Feature scaling is critical — unscaled features distort distances
- Irrelevant features hurt KNN more than most algorithms

**Choosing K:**
- Too small → overfitting, sensitive to noise
- Too large → underfitting, loses local patterns
- Starting point: K = √n where n is number of training samples
- Always use odd K for binary classification to avoid ties
- Try multiple values and compare performance on validation set

**Steps:**

1. Load, explore, clean data — same as other classifiers
2. Prepare features — feature selection matters more here than other models
3. Split data — train_test_split(X, y, test_size=0.2, random_state=42)
4. Scale features — critical for KNN, always scale
5. Train model:
   - model = KNeighborsClassifier(n_neighbors=5)
   - model.fit(X_train_scaled, y_train)
6. Evaluate — same as binary or multiclass classification
7. Tune K — try different values of n_neighbors and compare F1/accuracy

**Common issues and considerations:**

- **Slow predictions on large datasets:** KNN searches the entire training 
  set for every prediction. Not suitable for real-time applications with 
  large data.
- **Curse of dimensionality:** with many features, all points become 
  roughly equidistant and neighbors become meaningless. Use dimensionality 
  reduction first if you have many features.
- **Missing values:** KNN cannot handle missing values. Always clean data 
  before training.
- **Imbalanced classes:** majority vote is biased toward the dominant class. 
  Use class_weight='balanced' or adjust K.

## 6  - REGULARIZATION (Ridge, Lasso, ElasticNet)

**When to use:**
When linear regression shows signs of overfitting, unstable coefficients, 
or when multicollinearity is detected between features.

**When NOT to use:**
- Tree-based models (they don't use coefficients)
- When the dataset is small and clean with no multicollinearity

**Imports:**
```python
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import numpy as np
```

**Steps:**

1. Train a baseline LinearRegression first
   - Evaluate with RMSE and R²
   - Print coefficients to identify instability
   - This gives you a reference point to compare against

2. Check for multicollinearity
   - Plot correlation heatmap
   - Calculate VIF scores
   - Identify which features are problematic

3. Choose regularization type
   - Most features relevant, multicollinearity present → Ridge
   - Many features suspected irrelevant → Lasso
   - Unsure → ElasticNet

4. Find best alpha automatically:

   **Ridge:**
```python
   ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100])
   ridge_cv.fit(X_train_scaled, y_train)
   print(f"Best alpha: {ridge_cv.alpha_}")
   model = Ridge(alpha=ridge_cv.alpha_)
```

   **Lasso:**
```python
   lasso_cv = LassoCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100], cv=5)
   lasso_cv.fit(X_train_scaled, y_train)
   print(f"Best alpha: {lasso_cv.alpha_}")
   model = Lasso(alpha=lasso_cv.alpha_)
```

   **ElasticNet:**
```python
   # No ElasticNetCV with custom alphas, use manual cross-validation
   # l1_ratio: 0 = pure Ridge, 1 = pure Lasso
   model = ElasticNet(alpha=1.0, l1_ratio=0.5)
```

5. Train final model with best alpha
```python
   model.fit(X_train_scaled, y_train)
```

6. Evaluate and compare against baseline
```python
   y_pred = model.predict(X_test_scaled)
   mse = mean_squared_error(y_test, y_pred)
   rmse = np.sqrt(mse)
   r2 = r2_score(y_test, y_pred)
   print(f"RMSE: {rmse:.4f}")
   print(f"R²: {r2:.4f}")
```

7. Print coefficients and compare to baseline
```python
   coef_df = pd.DataFrame({
       'feature': feature_names,
       'coefficient': model.coef_
   })
   print(coef_df.sort_values('coefficient', ascending=False))
```

8. Run cross-validation to verify performance
```python
   scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
   print(f"Average R²: {scores.mean():.2f}")
```

**Interpret results:**
- Coefficients should be smaller and more stable than baseline
- Collinear feature pairs should have less extreme opposite values
- If Lasso sets coefficients to exactly zero, those features were irrelevant
- RMSE and R² may improve slightly or stay similar — the main gain 
  is stability and generalization, not always raw performance
- Cross-validation R² gives a more reliable estimate than a single test split

**Common issues and considerations:**

- Always scale before regularization — penalties are applied to 
  coefficient values which are affected by feature scale
- If RidgeCV or LassoCV picks the largest alpha you provided, extend 
  the range upward — optimal value may be outside your search range
- If it picks the smallest, extend the range downward
- ElasticNet has two parameters: alpha (overall penalty strength) and 
  l1_ratio (balance between Ridge and Lasso, 0=pure Ridge, 1=pure Lasso)
- Cross-validation score may differ from test set score on small 
  datasets — this is normal, CV gives the more reliable estimate

## 7. KMeans Clustering

**When to use:**
When you want to discover natural groupings in unlabelled data. Use when you
have no target variable and want to segment data into clusters.

**When NOT to use:**
- When you have labelled data — use a classifier instead
- When clusters are non-spherical or very different in size
- When the number of clusters is completely unknown and data gives no hints

**Imports:**
```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
```

**Key concepts:**
- Unsupervised — no labels, no train/test split, fit on the full dataset
- Algorithm: place k centroids randomly, assign each point to nearest centroid,
  recalculate centroids, repeat until convergence
- inertia_ = sum of squared distances from each point to its centroid (lower is better)
- Silhouette score = how tight clusters are and how separated from each other
  (higher is better, range -1 to 1)

**Steps:**

1. Prepare features
```python
X = df.drop('label_column', axis=1)  # drop any existing labels
```

2. Scale features — fit on full dataset since there is no train/test split
```python
scaler = StandardScaler()
scaled_X = scaler.fit_transform(X)
```

3. Find optimal k with elbow plot
```python
clusters = range(1, 10)
inertias = [KMeans(n_clusters=k, random_state=42).fit(scaled_X).inertia_ for k in clusters]
plt.plot(clusters, inertias, '-o')
plt.xlabel('Number of clusters k')
plt.ylabel('Inertia')
plt.title('Elbow plot')
plt.show()
# Look for the elbow — where the curve stops dropping sharply
```

4. Confirm with silhouette score
```python
kmeans_list = [KMeans(n_clusters=k, random_state=42).fit(scaled_X) for k in range(2, 10)]
silhouette_scores = [silhouette_score(scaled_X, km.labels_) for km in kmeans_list]
plt.plot(range(2, 10), silhouette_scores, 'o-')
plt.xlabel('k clusters')
plt.ylabel('Silhouette score')
plt.show()
# Higher silhouette score = better defined clusters
```

5. Fit final model with chosen k
```python
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(scaled_X)
labels = kmeans.labels_
centers = kmeans.cluster_centers_
```

6. Visualize (only meaningful for 2D data)
```python
df_plot = pd.DataFrame(scaled_X, columns=['x1', 'x2'])
df_plot['cluster'] = labels
sns.scatterplot(data=df_plot, x='x1', y='x2', hue='cluster', palette='tab10')
plt.scatter(centers[:, 0], centers[:, 1], s=200, marker='*', color='black', label='centroid')
plt.title('KMeans clusters')
plt.show()
```

**Common issues and considerations:**

- No train/test split — scale the full dataset at once
- Results depend on random initialization — always set random_state for reproducibility
- KMeans assumes spherical clusters of similar size — if clusters have irregular
  shapes consider DBSCAN instead
- Elbow plot alone is often ambiguous — always confirm with silhouette score
- High dimensional data makes distance measures less meaningful — consider
  dimensionality reduction before clustering

## 8. TF-IDF (Text Feature Extraction)

**When to use:**
When working with text data and needing to convert documents into numerical
vectors for use in ML models. Useful for text classification, similarity search,
and recommendation systems.

**When NOT to use:**
- When word order matters — TF-IDF treats text as a bag of words with no sequence
- When working with very short texts — too little context for IDF to be meaningful
- When semantic meaning matters — use word embeddings instead

**Imports:**
```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
```

**Key concepts:**
- TF (Term Frequency): how often a word appears in a document
- IDF (Inverse Document Frequency): how rare a word is across all documents —
  common words like "the" get low scores, rare distinctive words get high scores
- TF-IDF = TF × IDF — words that appear often in one document but rarely
  across all documents get the highest scores
- Output is a sparse matrix — most values are zero

**Steps:**

1. Prepare text data as a list or Series of strings
```python
documents = df['text_column'].tolist()
```

2. Fit and transform with TfidfVectorizer
```python
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)
# tfidf_matrix is sparse — shape is (n_documents, n_unique_words)
```

3. Inspect feature names
```python
feature_names = vectorizer.get_feature_names_out()
```

4. Convert to DataFrame for inspection
```python
tfidf_df = pd.DataFrame(tfidf_matrix.todense(), columns=feature_names)
```

5. Compute similarity between documents
```python
similarity_matrix = cosine_similarity(tfidf_matrix)
# similarity_matrix[i][j] = similarity between document i and document j
# 1.0 = identical, 0.0 = no overlap
```

6. Find most similar documents to a query document
```python
doc_index = 0  # index of the document to compare against
scores = list(enumerate(similarity_matrix[doc_index]))
scores = sorted(scores, key=lambda x: x[1], reverse=True)
top_matches = scores[1:6]  # skip index 0 (itself)
```

**Common issues and considerations:**

- fit_transform on training data only — use transform on new data to avoid leakage
- TfidfVectorizer automatically lowercases and removes punctuation by default
- Very common words add noise — use max_df to ignore words appearing in more
  than a set proportion of documents: TfidfVectorizer(max_df=0.9)
- Very rare words add noise — use min_df to ignore words appearing in fewer
  than a set number of documents: TfidfVectorizer(min_df=2)
- Output is sparse — use .todense() only for small datasets or inspection,
  most sklearn models accept sparse matrices directly

## 9. PyTorch Neural Network (Regression)

**When to use:**
When the relationship between features and target is non-linear and too complex
for linear models. Needed when full control over architecture and training is
required.

**When NOT to use:**
- Small datasets with simple linear relationships — use Ridge or Linear Regression
- When sklearn's pipeline tools are sufficient for the task

**Imports:**
```python
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
```

**Key concepts:**
- Tensors: PyTorch's data format, equivalent to numpy arrays but GPU-compatible
  and capable of tracking gradients for backpropagation
- nn.Sequential: container that holds layers in order — data flows top to bottom
- nn.Linear(in, out): one fully connected layer with `out` neurons each receiving
  `in` inputs. Input size must match previous layer's output size exactly
- nn.ReLU(): activation function applied between layers — no learnable parameters
- Training loop: the explicit cycle of forward pass, loss computation,
  backpropagation and weight update that sklearn hides inside .fit()

**Steps:**

1. Convert scaled numpy arrays to tensors
```python
device = 'cuda' if torch.cuda.is_available() else 'cpu'
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)
```
   - Always use dtype=torch.float32
   - unsqueeze(1) turns y shape (n,) into (n, 1) — required to match model output shape

2. Create DataLoader for mini-batch training
```python
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
```
   - TensorDataset keeps X and y paired during shuffling
   - batch_size=32 is a sensible default

3. Define model — input size = number of features, output size = 1 for regression
```python
model = nn.Sequential(
    nn.Linear(n_features, 64),
    nn.ReLU(),
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
).to(device)
```

4. Define loss function and optimizer
```python
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
```

5. Training loop
```python
n_epochs = 30
for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        y_pred  = model(X_batch)
        loss    = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    mean_loss = total_loss / len(train_loader)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {mean_loss:.4f}")
```
   - model.train() sets training mode at the start of each epoch
   - Batches must be moved to the same device as the model
   - zero_grad() clears gradients after each batch — skipping it corrupts updates
   - Print every N epochs: (epoch + 1) % N == 0

6. Evaluate
```python
model.eval()
with torch.no_grad():
    y_pred    = model(X_test_tensor.to(device))
    test_loss = criterion(y_pred, y_test_tensor.to(device))
    rmse      = torch.sqrt(test_loss)
print(f"Test RMSE: {rmse.item():.4f}")
```
   - model.eval() sets evaluation mode
   - torch.no_grad() disables gradient tracking — saves memory

**Common issues and considerations:**

- Always scale data before converting to tensors
- GPU is not always faster — for small tabular datasets CPU may be quicker
  due to data transfer overhead. GPU advantage appears with large models and images
- For tabular data, wider layers tend to outperform deeper narrow ones
- If loss is not decreasing, try a lower learning rate or switch to Adam
- Hidden layer count and neuron count are hyperparameters — input and output
  sizes are fixed by the data and must not be changed

### 9b. PyTorch Neural Network (Classification)

**Differences from regression:**
- y tensors use dtype=torch.long, no unsqueeze
- Output layer size = number of classes (e.g. 10 for Fashion-MNIST)
- Loss function: nn.CrossEntropyLoss() — applies softmax internally,
  never add softmax to the model output
- No activation function after the output layer

**Changes to the pipeline:**

Tensors:
```python
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.long)
```

Loss function:
```python
criterion = nn.CrossEntropyLoss()
```

Evaluation:
```python
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensor.to(device))
    predicted_classes = torch.argmax(y_pred, dim=1)
    accuracy = (predicted_classes == y_test_tensor.to(device)).float().mean()
print(f"Accuracy: {accuracy.item()*100:.2f}%")
```
- torch.argmax(y_pred, dim=1) picks the highest scoring class per sample
- Accuracy = proportion correct, multiply by 100 for percentage